In [ ]:
import pandas as pd
import io

filepath = ""
df = pd.read_csv(filepath)  

print("File Loaded:", filepath)
print("Shape:", df.shape)
df.head()

Saving uav_retrieval_results_with_full_energy_logging.csv to uav_retrieval_results_with_full_energy_logging.csv
File Loaded: uav_retrieval_results_with_full_energy_logging.csv
Shape: (342, 41)


,Dataset,Pipeline,Anchor_text,Requirement,Rank,Latency_s,timestamp,project_name,run_id,experiment_id,...,gpu_model,longitude,latitude,ram_total_size,tracking_mode,on_cloud,pue,wue,cloud_provider,cloud_region
0,uav,KAG,"When requested, the _VehicleCore_ shall transi...",If a command is issued by the _VaadinUI_ for a...,1,0.066245,2026-02-25T04:37:54,KAG_retrieval,f873d928-2543-4e6e-b23d-449c689acec5,uav,...,Apple M4,-114.0718,51.023,16.0,process,N,1.0,0.0,NaN,NaN
1,uav,KAG,"When requested, the _VehicleCore_ shall transi...","If requested, the _SingleUAVFlightPlanSchedule...",2,0.066245,2026-02-25T04:37:54,KAG_retrieval,f873d928-2543-4e6e-b23d-449c689acec5,uav,...,Apple M4,-114.0718,51.023,16.0,process,N,1.0,0.0,NaN,NaN
2,uav,RAG,"When requested, the _VehicleCore_ shall transi...",The _VehicleCore_ shall assign an ID to each a...,1,0.184475,2026-02-25T04:37:59,RAG_retrieval,70e595d4-7c70-4da9-9e3e-bdefe52c1ce4,uav,...,Apple M4,-114.0718,51.023,16.0,process,N,1.0,0.0,NaN,NaN
3,uav,KAG,The _VehicleCore_ shall assign a unique ID to ...,The _VehicleCore_ shall assign an ID to each a...,1,0.028621,2026-02-25T04:38:02,KAG_retrieval,194b5892-00cd-42c6-8a0b-8a06a42e8089,uav,...,Apple M4,-114.0718,51.023,16.0,process,N,1.0,0.0,NaN,NaN
4,uav,KAG,The _VehicleCore_ shall assign a unique ID to ...,If a scheduled flight is not manually assigned...,2,0.028621,2026-02-25T04:38:02,KAG_retrieval,194b5892-00cd-42c6-8a0b-8a06a42e8089,uav,...,Apple M4,-114.0718,51.023,16.0,process,N,1.0,0.0,NaN,NaN


In [ ]:
numeric_cols = [
    "emissions",
    "energy_consumed",
    "cpu_energy",
    "gpu_energy",
    "ram_energy",
    "Latency_s"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["emissions", "energy_consumed"])

print("Cleaned Shape:", df.shape)

Cleaned Shape: (342, 41)


In [ ]:

grouped = df.groupby(["Dataset", "Pipeline"]).agg(
    Retrievals=("Requirement", "count"),
    Total_Energy_kWh=("energy_consumed", "sum"),
    Total_Carbon_kgCO2e=("emissions", "sum"),
    Avg_Energy_per_Retrieval_kWh=("energy_consumed", "mean"),
    Avg_Carbon_per_Retrieval_kgCO2e=("emissions", "mean"),
    Total_CPU_Energy_kWh=("cpu_energy", "sum"),
    Total_GPU_Energy_kWh=("gpu_energy", "sum"),
    Total_RAM_Energy_kWh=("ram_energy", "sum"),
    Total_Latency_s=("Latency_s", "sum")
).reset_index()

grouped = grouped.round(8)

grouped

,Dataset,Pipeline,Retrievals,Total_Energy_kWh,Total_Carbon_kgCO2e,Avg_Energy_per_Retrieval_kWh,Avg_Carbon_per_Retrieval_kgCO2e,Total_CPU_Energy_kWh,Total_GPU_Energy_kWh,Total_RAM_Energy_kWh,Total_Latency_s
0,uav,KAG,228,0.000008,0.000005,4.000000e-08,2.000000e-08,7.500000e-07,0.000003,0.000004,4.749930
1,uav,RAG,114,0.000008,0.000005,7.000000e-08,5.000000e-08,8.500000e-07,0.000002,0.000006,6.868249
